# VFD Deep Dive: Vigesimal Feature Decomposition

This notebook explores the VFD encoder in detail, including:
- How multi-scale decomposition works
- Component modes (full, lite, bars_dots)
- Visualization of Maya numbers
- Feature analysis and inverse transforms

In [ ]:
import numpy as np
from maya_encoding import VFDEncoder, maya_decompose, to_vigesimal, to_bars_dots
from maya_encoding.core.vigesimal import auto_n_levels, maya_encode_array

## 1. Multi-Scale Decomposition

The vigesimal system decomposes numbers into hierarchical levels:
- **Level 0** (ones): captures values 0-19
- **Level 1** (twenties): captures values 0-19 × 20
- **Level 2** (four-hundreds): captures values 0-19 × 400

Each level is further decomposed into **bars** (÷5) and **dots** (%5).

In [ ]:
# Demonstrate multi-scale decomposition
numbers = [0, 5, 19, 20, 100, 347, 399, 400, 8000]

print(f"{'Number':>6} | {'Base-20 digits':>15} | {'Bars':>10} | {'Dots':>10} | Levels")
print("-" * 70)
for n in numbers:
    info = maya_decompose(n)
    digits_str = str(info['digits'])
    bars_str = str(info['bars'])
    dots_str = str(info['dots'])
    print(f"{n:>6} | {digits_str:>15} | {bars_str:>10} | {dots_str:>10} | {info['n_levels']}")

In [ ]:
# Auto level detection
test_values = [0, 1, 19, 20, 399, 400, 7999, 8000, 160000]
for v in test_values:
    levels = auto_n_levels(v)
    max_val = 20**levels - 1
    print(f"  {v:>7} → {levels} levels (max representable: {max_val})")

## 2. Component Modes

VFDEncoder supports three component modes:
- **full**: digit + bars + dots per level (3 features × n_levels)
- **lite**: only digits per level (1 feature × n_levels)
- **bars_dots**: bars + dots per level (2 features × n_levels)

In [ ]:
X = np.array([[347]])

for mode in ["full", "lite", "bars_dots"]:
    enc = VFDEncoder(n_levels=2, components=mode, normalize=False)
    result = enc.fit_transform(X)
    names = enc.get_feature_names_out()
    print(f"\n{mode.upper()} mode ({len(names)} features):")
    for name, val in zip(names, result[0]):
        print(f"  {name}: {val}")

## 3. Normalization

When `normalize=True`, each component is scaled to [0, 1]:
- Digits: divided by 19
- Bars: divided by 3
- Dots: divided by 4

In [ ]:
X = np.array([[0], [10], [19], [347]])

enc_raw = VFDEncoder(n_levels=2, normalize=False)
enc_norm = VFDEncoder(n_levels=2, normalize=True)

raw = enc_raw.fit_transform(X)
normed = enc_norm.fit_transform(X)

print("Raw vs Normalized (first sample, value=0):")
for name, r, n in zip(enc_raw.get_feature_names_out(), raw[0], normed[0]):
    print(f"  {name}: {r:.0f} → {n:.4f}")

print(f"\nNormalized range: [{normed.min():.4f}, {normed.max():.4f}]")

## 4. Inverse Transform

VFDEncoder supports inverse_transform to reconstruct original values.

In [ ]:
X = np.array([[0], [7], [20], [347], [399]])

for mode in ["full", "lite"]:
    enc = VFDEncoder(n_levels=2, components=mode, normalize=False)
    encoded = enc.fit_transform(X)
    reconstructed = enc.inverse_transform(encoded)
    
    print(f"\n{mode} mode roundtrip:")
    for orig, recon in zip(X.ravel(), reconstructed.ravel()):
        match = '✓' if orig == recon else '✗'
        print(f"  {orig:>4} → encode → decode → {recon:>4} {match}")

## 5. Vectorized Performance

The encoder uses vectorized numpy operations for efficient batch processing.

In [ ]:
import time

sizes = [100, 1000, 10000, 50000]
enc = VFDEncoder(n_levels=3, components="full")

print(f"{'N samples':>10} | {'Time (ms)':>10} | {'Throughput':>15}")
print("-" * 45)
for n in sizes:
    X = np.random.randint(0, 8000, size=(n, 3)).astype(float)
    start = time.perf_counter()
    enc.fit_transform(X)
    elapsed = (time.perf_counter() - start) * 1000
    throughput = n / (elapsed / 1000)
    print(f"{n:>10,} | {elapsed:>9.1f} | {throughput:>12,.0f}/s")

## 6. Visualization

The library includes text-based and matplotlib visualization of Maya numbers.

In [ ]:
from maya_encoding.visualization.glyphs import render_maya_text

for n in [0, 7, 19, 20, 347, 8000]:
    text = render_maya_text(n)
    print(f"\n{n}:")
    print(text)